# OP2 Reader — Test Notebook

In [ ]:
# Setup — reload module and open file
import sys
for mod in list(sys.modules.keys()):
    if "op2_native" in mod:
        del sys.modules[mod]

sys.path.insert(0, r"c:\Users\evanc\Dropbox\Coding\Python\OP2_reader")
from op2_native import OP2

OP2_FILE = r"c:\Users\evanc\Dropbox\Coding\Python\OP2_reader\testing-015.op2"
op2 = OP2(OP2_FILE)
print("Loaded:", OP2_FILE)


In [ ]:
## 1 — File Summary
# All records scanned from the binary file.

In [ ]:
summary = op2.summary()
print(f"{len(summary)} records total")
summary


In [ ]:
## 2 — Displacements (OUGV1)
# Columns: GRID, CP (output coord system), DX, DY, DZ (translations), RX, RY, RZ (rotations)

In [ ]:
displ = op2.displacements()
for sc, df in displ.items():
    print(f"Subcase {sc}: {len(df)} rows")
    display(df.head(10))


In [ ]:
## 3 — Element Stresses (OES1X1 shell)
# Centroid-only:  EID, FD1, SX1, SY1, TXY1, ANG1, MAJOR1, MINOR1, VM1,
#                     FD2, SX2, SY2, TXY2, ANG2, MAJOR2, MINOR2, VM2
# With corners:   EID, GRID (0=centroid), FD1..VM2  →  5 rows/element
# VM1/VM2 = Von Mises (or Max Shear, depending on Nastran STRESS case control)

In [ ]:
stresses = op2.stresses()
for sc, df in stresses.items():
    print(f"Subcase {sc}: {len(df)} rows")
    display(df)


In [ ]:
## 4 — Element Forces (OEF1)
# CQUAD4/CTRIA3: EID, NX, NY, NXY (membrane), MX, MY, MXY (bending), QX, QY (transverse shear)
# CBAR/CBEAM:   EID, BM1A, BM2A, BM1B, BM2B, TS1, TS2, AF, TRQ

In [ ]:
eforces = op2.element_forces()
for sc, df in eforces.items():
    print(f"Subcase {sc}: {len(df)} rows")
    display(df.head(10))


In [ ]:
## 5 — SPC Constraint Forces (OQG1)

In [ ]:
spc = op2.spc_forces()
for sc, df in spc.items():
    nonzero = df[(df[["FX","FY","FZ","MX","MY","MZ"]].abs() > 0).any(axis=1)]
    print(f"Subcase {sc}: {len(df)} rows total, {len(nonzero)} nonzero")
    display(nonzero.head(20))


In [ ]:
## 6 — Applied Loads (OPG1)

In [ ]:
try:
    loads = op2.applied_loads()
    if loads:
        for sc, df in loads.items():
            print(f"Subcase {sc}: {len(df)} rows")
            display(df.head(10))
    else:
        print("No OPG1 tables found in this file.")
except Exception as e:
    print(f"Error: {e}")


In [ ]:
## 7 — Strains (OSTR1)

In [ ]:
try:
    strains = op2.strains()
    if strains:
        for sc, df in strains.items():
            print(f"Subcase {sc}: {len(df)} rows")
            display(df.head(10))
    else:
        print("No OSTR1 tables found in this file.")
except Exception as e:
    print(f"Error: {e}")


In [ ]:
## 8 — Solid / Bar Stresses
# Returns empty for this file (no solid/bar elements), but confirms the API works.

In [ ]:
solid = op2.solid_stresses()
bars  = op2.bar_stresses()
print(f"solid_stresses subcases : {sorted(solid.keys())}")
print(f"bar_stresses   subcases : {sorted(bars.keys())}")
if not solid and not bars:
    print("(none — no solid/bar elements in this file)")


In [ ]:
## 8b — Eigenvalues (LAMA)
# Available for modal (SOL 103) and buckling (SOL 105) analyses.
# Columns: MODE, ORDER, EIGENVALUE (ω², rad²/s²), RADIANS (ω, rad/s),
#          CYCLES (f, Hz), GENM (generalised mass), GENSTIF (generalised stiffness)
eigs = op2.eigenvalues()
if eigs:
    for sc, df in eigs.items():
        print(f"Subcase {sc}: {len(df)} modes")
        display(df)
else:
    print("No LAMA table found (this is a static analysis file).")

In [ ]:
## 9 — Summary Table
import pandas as pd

all_results = {
    "displacements":      op2.displacements(),
    "stresses":           op2.stresses(),
    "stresses_corners":   op2.stresses_with_corners(),
    "solid_stresses":     op2.solid_stresses(),
    "bar_stresses":       op2.bar_stresses(),
    "element_forces":     op2.element_forces(),
    "spc_forces":         op2.spc_forces(),
    "applied_loads":      op2.applied_loads(),
    "strains":            op2.strains(),
    "eigenvalues":        op2.eigenvalues(),
}

rows = []
for name, d in all_results.items():
    if d:
        for sc, df in d.items():
            rows.append({"result": name, "subcase": sc, "rows": len(df), "cols": len(df.columns)})
    else:
        rows.append({"result": name, "subcase": "-", "rows": 0, "cols": 0})

# Grid Point Weight Generator (OGPWG) — single result, not subcase-keyed
gw = op2.grid_weight()
if gw:
    rows.append({"result": "grid_weight", "subcase": "-", "rows": 1, "cols": len(gw["summary"].columns)})
else:
    rows.append({"result": "grid_weight", "subcase": "-", "rows": 0, "cols": 0})

pd.DataFrame(rows)

In [ ]:
## 10 — Grid Point Weight (OGPWG)
# mass, centre of gravity, and inertia matrix from the OGPWG table.

In [ ]:
gw = op2.grid_weight()
if gw:
    print(f"Total mass  : {gw['mass']:.6g}")
    print(f"CG (X, Y, Z): {[round(v, 6) for v in gw['cg']]}")
    display(gw["summary"])
else:
    print("No OGPWG table found in this file.")

In [ ]:
## 11 — Corner Stresses (OES1X1 with corner nodes)
# 5 rows per CQUAD4: GRID=0 is centroid, GRID!=0 are the 4 corner node IDs.
sc_corners = op2.stresses_with_corners()
for sc, df in sc_corners.items():
    centroids = df[df["GRID"] == 0]
    corners   = df[df["GRID"] != 0]
    print(f"Subcase {sc}: {len(df)} rows  ({len(centroids)} centroids + {len(corners)} corner rows)")
    display(df.head(10))

In [ ]:
## 12 — find_by_eid  &  to_csv

# -- 12a: look up all results for a single element --
EID_QUERY = 1
hits = op2.find_by_eid(EID_QUERY)
print(f"Results found for EID={EID_QUERY}:")
for result_name, df in hits.items():
    print(f"  {result_name}: {len(df)} row(s)")
    display(df)

# -- 12b: export every result table to CSV --
import tempfile, os
out_dir = r"c:\Users\evanc\Dropbox\Coding\Python\OP2_reader\csv_export"
written = op2.to_csv(out_dir)
print(f"\nCSV files written to: {out_dir}")
for fname, fpath in sorted(written.items()):
    size_kb = os.path.getsize(fpath) / 1024
    print(f"  {fname:<35s}  {size_kb:7.1f} kB")

In [ ]:
# §13 — Performance: cache speedup
import time, importlib, op2_native.reader as _r
importlib.reload(_r)
from op2_native.reader import OP2

fresh = OP2(OP2_FILE)

# First call — cold (decode from bytes)
t0 = time.perf_counter()
_ = fresh.stresses()
cold_ms = (time.perf_counter() - t0) * 1000

# Second call — warm (from cache)
t0 = time.perf_counter()
_ = fresh.stresses()
warm_ms = (time.perf_counter() - t0) * 1000

print(f"stresses()  cold: {cold_ms:.2f} ms")
print(f"stresses()  warm: {warm_ms:.2f} ms  ({cold_ms/max(warm_ms,0.001):.0f}× speedup)")

# Confirm cache is populated
print(f"\nCache keys: {list(fresh._cache.keys())}")
fresh.clear_cache()
print(f"After clear_cache(): {list(fresh._cache.keys())}")

In [ ]:
# §14 — find_by_grid(grid_id): nodal results lookup
GRID_QUERY = 51   # pick any node ID from the model

grid_hits = op2.find_by_grid(GRID_QUERY)
print(f"find_by_grid({GRID_QUERY}) found results in: {list(grid_hits.keys())}\n")
for name, df in grid_hits.items():
    print(f"--- {name} ({len(df)} rows) ---")
    display(df.head(6))

In [ ]:
## 15 — OP2 repr  &  Results object

# The OP2 repr now shows a summary of all non-empty result types
print(repr(op2))
print()

# Results gives flat attribute access to all tables for one subcase
from op2_native import Results
r = op2.results(subcase=1)
print(repr(r))
print()

# Access individual tables directly as attributes
print("Stresses (first 3 rows):")
display(r.stresses.head(3))

print("Element forces (first 3 rows):")
display(r.element_forces.head(3))

In [ ]:
## 16 — Plotly Visualisations
from op2_native import plots

r = op2.results(subcase=1)

# Von Mises stress — max of both fibers, coloured bar chart sorted by EID
plots.plot_vm_stress(r.stresses, fiber="max")

In [ ]:
# Displacement magnitude scatter — each node coloured by |U|
plots.plot_displacement_magnitude(r.displacements, component="mag")

In [ ]:
# Element force NX (membrane x) — diverging red-blue colorscale
plots.plot_element_forces(r.element_forces, component="NX")

In [ ]:
# VM1 stress histogram with mean line
plots.plot_stress_histogram(r.stresses, column="VM1", bins=40)

## §17 — envelope(), describe(), and new plots

In [ ]:
# Reload modules so changes to reader.py and plots.py are picked up
import importlib, op2_native, op2_native.reader, op2_native.plots
importlib.reload(op2_native.plots)
importlib.reload(op2_native.reader)
importlib.reload(op2_native)

from op2_native import OP2
op2 = OP2(OP2_FILE)
r   = op2.results(1)
print("Modules reloaded OK")

In [ ]:
# envelope() — worst-case VM1 stress across all subcases
env = op2.envelope("stresses", "VM1", "absmax")
print(f"Envelope shape: {env.shape}")
print(f"Governing column range: {env['VM1'].min():.4g} → {env['VM1'].max():.4g}")
display(env.head(10))

In [ ]:
# describe() — statistical summary of all non-empty result tables
desc = op2.describe()
print(f"describe() shape: {desc.shape}")
display(desc.round(4))

In [ ]:
# Top-20 stressed elements — horizontal bar chart with EID labels
op2_native.plots.plot_top_n_stress(r.stresses, n=20, fiber="max")

In [ ]:
# Principal stress comparison: MAJOR vs MINOR vs VM (fiber 1, top 50 elements by VM)
op2_native.plots.plot_principal_stress(r.stresses, fiber="1", max_elements=50)

In [ ]:
# Polar scatter of in-plane element forces: NX at 0°, NY at 90°, NXY at 45°
op2_native.plots.plot_forces_polar(r.element_forces)

## §18 — subcases(), eqexin(), plot_stress_summary()

In [ ]:
import importlib, op2_native, op2_native.reader, op2_native.plots
importlib.reload(op2_native.plots)
importlib.reload(op2_native.reader)
importlib.reload(op2_native)
from op2_native import OP2
op2 = OP2(OP2_FILE)
r   = op2.results(1)
print("Reloaded")

In [ ]:
# subcases() — cross-tab of result type vs subcase ID
sc_table = op2.subcases()
print(sc_table.to_string())
display(sc_table)

In [ ]:
# eqexin() — GRID to DOF-pointer mapping
eq = op2.eqexin()
print(f"EQEXIN: {len(eq)} rows, GRID range {eq['GRID'].min()}–{eq['GRID'].max()}")
print(f"Constrained grids (EQTYPE==0): {(eq['EQTYPE']==0).sum()}")
display(eq.head(10))

In [ ]:
# Stress component summary — 4×2 histogram grid for all components
op2_native.plots.plot_stress_summary(r.stresses, fiber="1")

## §19 — Multi-record data fix verification

**Problem (fixed 2026-03-30):** All three data decoders (`decode_oes1x1_shell`, `decode_oes1x1_shell_corners`, `decode_oef1`) only read the **first** Fortran data record per table, silently dropping the remaining records. For a ~3,661-element model the OES1X1 table has 5 data records and OEF1X has 3, so ~80% of elements were missing from all outputs.

**Fix:** `collect_data_records_after()` + `load_data_bytes()` added to `oes_peek.py`. Both shell decoders and the force decoder now concatenate all contiguous data records before decoding.

In [ ]:
# §19a — Verify full element counts after multi-record fix
import sys
for mod in list(sys.modules.keys()):
    if "op2_native" in mod:
        del sys.modules[mod]

from op2_native import OP2
op2 = OP2(OP2_FILE)
r = op2.results(1)

st = r.stresses
sc = r.stresses_corners
ef = r.element_forces

print(f"stresses()        rows: {len(st):>6,}   unique EIDs: {st['EID'].nunique():>5,}")
print(f"stresses_corners  rows: {len(sc):>6,}   unique EIDs: {sc['EID'].nunique():>5,}")
print(f"element_forces()  rows: {len(ef):>6,}   unique EIDs: {ef['EID'].nunique():>5,}")
print()
print(f"stresses   attrs: {st.attrs.get('all_data_records')}")
print(f"forces     attrs: {ef.attrs.get('all_data_records')}")


## §19b — Next steps

- **`oes_bar.py`** — `decode_oes_bar` still uses `first_stress_record_after` + single `rec.data`. Update to `load_data_bytes` (same pattern, but bar models are often small so may not be urgent).
- **`lama.py`** — `_first_data_record_after` is its own local helper; verify that eigenvalue tables never span multiple records (they typically don't for modal analyses).
- **Solid elements** — no solid-element OES decoder exists yet; if needed, add `oes1x1_solid.py` following the same `load_data_bytes` pattern.
- **OUG / grid-point results** — already single-record in typical models; no change needed unless very large grids are encountered.
- **Test with pyNastran** — cross-check EID sets and stress values against `pyNastran.op2.OP2` as a reference implementation.

In [ ]:
EIDS_TO_CHECK = [417, 418, 419] # CBEAM element IDs
GRIDS_TO_CHECK = [1, 2, 3] # node IDs

import sys, warnings

from matplotlib.pyplot import title
warnings.filterwarnings("ignore")
for mod in list(sys.modules.keys()):
    if "op2_native" in mod:
        del sys.modules[mod]

from op2_native import OP2
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.6e}".format)

CYL_OP2 = r"run_files/model/cylinder-0000.op2"
cyl = OP2(CYL_OP2)
print("Loaded:", CYL_OP2)

all_sc = sorted(
set(cyl.displacements()) | set(cyl.bar_stresses()) |
set(cyl.element_forces()) | set(cyl.bush_stresses()) |
set(cyl.bush_forces()) | set(cyl.solid_stresses())
)
print(f"Subcases found: {all_sc}\n")



def section(title):
    print(f"\n{'═'*72}\n {title}\n{'═'*72}")

for sc in all_sc:
    print(f"\n\n{'#'*72}")
    print(f" SUBCASE {sc}")
    print(f"{'#'*72}")

    # ── Displacements ─────────────────────────────────────────────────────────
    disp = cyl.displacements().get(sc)
    if disp is not None:
        d = disp[disp["GRID"].isin(GRIDS_TO_CHECK)]
        if len(d):
            section("DISPLACEMENTS  (T1/T2/T3 = translations,  R1/R2/R3 = rotations)")
            display(d.set_index("GRID").drop(columns=["CP"], errors="ignore"))

    # ── CBEAM stresses ────────────────────────────────────────────────────────
    bstr = cyl.bar_stresses().get(sc)
    if bstr is not None:
        b = bstr[bstr["EID"].isin(EIDS_TO_CHECK)]
        if len(b):
            section("CBEAM STRESSES  (SXC/SXD/SXE/SXF at 4 fiber corners,  SMAX/SMIN)")
            display(b.set_index(["EID", "GRID"]))

    # ── CBEAM forces ──────────────────────────────────────────────────────────
    ef = cyl.element_forces().get(sc)
    if ef is not None:
        e = ef[ef["EID"].isin(EIDS_TO_CHECK)]
        if len(e):
            section("CBEAM FORCES  (BM1/BM2 = bending moments,  WS1/WS2 = web shears,  AF = axial,  TRQ = torque)")
            display(e.set_index(["EID", "GRID"]))

    # ── CBUSH stresses ────────────────────────────────────────────────────────
    bush_s = cyl.bush_stresses().get(sc)
    if bush_s is not None:
        bs = bush_s[bush_s["EID"].isin(EIDS_TO_CHECK)]
        if len(bs):
            section("CBUSH STRESSES  (EX/EY/EZ = axial/shear,  ETX/ETY/ETZ = rotational)")
            display(bs.set_index("EID"))

    # ── CBUSH forces ──────────────────────────────────────────────────────────
    bush_f = cyl.bush_forces().get(sc)
    if bush_f is not None:
        bf = bush_f[bush_f["EID"].isin(EIDS_TO_CHECK)]
        if len(bf):
            section("CBUSH FORCES  (FX/FY/FZ = forces,  MX/MY/MZ = moments)")
            display(bf.set_index("EID"))

    # ── CTETRA stresses ───────────────────────────────────────────────────────
    sol = cyl.solid_stresses().get(sc)
    if sol is not None:
        ss = sol[sol["EID"].isin(EIDS_TO_CHECK)]
        if len(ss):
            section("CTETRA STRESSES  (GRID=0 → centroid,  GRID!=0 → corner node)")
            display(ss.set_index(["EID", "GRID"]))

    print("\n\nDone. Adjust EIDS_TO_CHECK / GRIDS_TO_CHECK at the top and re-run the two cells.")

In [ ]:
## §20 — Nonlinear Results (`cylinder-0001.op2`)

# This section exercises all result types on the nonlinear (SOL 106) run file.
# The model contains `CBEAM` + `CTETRA` elements (no `CBUSH`), so `bush_stresses` / `bush_strains` return empty dicts for this file.
# Linear-equivalent strains (`bar_strains`, `solid_strains`) are also decoded from the `OSTR1X` table.


In [1]:
# §20a — Load the NL file and show available results
import importlib, op2_native
importlib.reload(op2_native.reader); from op2_native.reader import OP2

NL_OP2 = 'run_files/model/cylinder-0001.op2'
nl = OP2(NL_OP2)

print("All tables (includes metadata blocks):")
print(nl.table_names(results_only=False))
print()
print("Result tables only:")
print(nl.table_names(results_only=True))


All tables (includes metadata blocks):
['NX2412', 'PVT0', 'PVT', 'CASECC', 'CASECC1', 'EQEXINS', 'EQEXIN', 'OGPWG', 'OGPWG', 'OESNLXR', 'OQG1', 'OUGV1', 'OEF1X', 'OES1X1', 'OSTR1X']

Result tables only:
['OGPWG', 'OGPWG', 'OESNLXR', 'OQG1', 'OUGV1', 'OEF1X', 'OES1X1', 'OSTR1X']


In [2]:
# §20b — Nonlinear bar stresses (OESNLXR CBEAM)
nl_bstr = nl.nl_bar_stresses()
for sc, df in nl_bstr.items():
    print(f"nl_bar_stresses sc={sc}: {len(df)} rows, {df['EID'].nunique()} elements")
nl_bstr[1].head(4)


nl_bar_stresses sc=1: 3328 rows, 416 elements
nl_bar_stresses sc=2: 3328 rows, 416 elements


,EID,GRID,FIBER,STRESS,EQ_STRESS,TOTAL_STRAIN,EFF_STRAIN_PLAS,EFF_CREEP
0,1,7573,C,2.554456,2.554456,1.277228e-07,0.0,0.0
1,1,7573,D,3.109892,3.109892,1.554946e-07,0.0,0.0
2,1,7573,E,2.554992,2.554992,1.277496e-07,0.0,0.0
3,1,7573,F,1.999557,1.999557,9.997783e-08,0.0,0.0


In [3]:
# §20c — Nonlinear solid stresses (OESNLXR CTETRA)
nl_sstr = nl.nl_solid_stresses()
for sc, df in nl_sstr.items():
    print(f"nl_solid_stresses sc={sc}: {len(df)} rows, {df['EID'].nunique()} elements")
# Show centroid rows only (GRID == 0)
nl_sstr[1][nl_sstr[1]['GRID'] == 0].head(4)


nl_solid_stresses sc=1: 24290 rows, 4858 elements
nl_solid_stresses sc=2: 24290 rows, 4858 elements


,EID,GRID,SX,SY,SZ,SXY,SYZ,SZX,EQ_STRESS,EFF_STRAIN_PLAS,EFF_CREEP,EX,EY,EZ,EXY,EYZ,EZX
0,1057,0,1.122189,6.152842,1.038326,-0.441499,0.876099,-0.013436,5.350173,0.0,0.0,-6.254483e-08,2.719936e-07,-6.812173e-08,-5.871939e-08,1.165211e-07,-1.787010e-09
5,1058,0,4.238459,2.683930,4.079813,1.418334,1.448172,2.538006,5.817745,0.0,0.0,1.003212e-07,-3.054979e-09,8.977121e-08,1.886384e-07,1.926068e-07,3.375548e-07
10,1059,0,2.570260,1.760756,4.559177,-0.903540,-1.794571,2.560980,6.165030,0.0,0.0,2.423411e-08,-2.959794e-08,1.564971e-07,-1.201708e-07,-2.386779e-07,3.406103e-07
15,1060,0,1.689896,2.426680,4.036803,-1.010507,2.419529,-1.735337,5.829372,0.0,0.0,-2.215265e-08,2.684345e-08,1.339166e-07,-1.343975e-07,3.217973e-07,-2.307999e-07


In [4]:
# §20d — Linear-equivalent bar strains (OSTR1X CBEAM) — NL file
nl_bs = nl.bar_strains()
for sc, df in nl_bs.items():
    print(f"bar_strains (NL) sc={sc}: {len(df)} rows, {df['EID'].nunique()} elements")
nl_bs[1].head(4)


bar_strains (NL) sc=1: 832 rows, 416 elements
bar_strains (NL) sc=2: 832 rows, 416 elements


,EID,GRID,SD,SXC,SXD,SXE,SXF,SMAX,SMIN,MS_T,MS_C
0,1,7573,0.0,1.277228e-07,1.554946e-07,1.277496e-07,9.997783e-08,1.554946e-07,9.997783e-08,1.401298e-45,1.401298e-45
1,1,225,1.0,1.281084e-07,9.972664e-08,1.273640e-07,1.557458e-07,1.557458e-07,9.972664e-08,1.401298e-45,1.401298e-45
2,2,7573,0.0,1.168698e-07,1.022666e-07,1.387414e-07,1.533447e-07,1.533447e-07,1.022666e-07,1.401298e-45,1.401298e-45
3,2,226,1.0,1.383857e-07,1.535917e-07,1.172256e-07,1.020196e-07,1.535917e-07,1.020196e-07,1.401298e-45,1.401298e-45


In [5]:
# §20e — Linear-equivalent solid strains (OSTR1X CTETRA) — NL file
nl_ss = nl.solid_strains()
for sc, df in nl_ss.items():
    print(f"solid_strains (NL) sc={sc}: {len(df)} rows, {df['EID'].nunique()} elements")
nl_ss[1].head(4)


solid_strains (NL) sc=1: 19432 rows, 4858 elements
solid_strains (NL) sc=2: 19432 rows, 4858 elements


,EID,GRID,SX,SY,SZ,SXY,SYZ,SZX,VM
0,1057,529,-6.097837e-08,-8.308245e-08,2.781773e-07,-8.658311e-09,-1.615073e-07,-8.755641e-09,2.518280e-07
1,1057,530,-5.164954e-08,-4.301470e-08,2.437082e-07,-1.599971e-08,-1.257997e-07,2.198005e-08,2.078289e-07
2,1057,531,-7.261163e-08,-9.727136e-08,2.837017e-07,-1.623900e-08,-8.834735e-08,1.957918e-08,2.518316e-07
3,1057,532,-7.234757e-08,-3.245602e-08,2.731325e-07,3.629215e-09,-1.882794e-07,2.401281e-08,2.442178e-07


In [6]:
# §20f — Bar / solid / bush strains on LINEAR file
lin = OP2('run_files/model/cylinder-0000.op2')
lin_bs = lin.bar_strains()
lin_ss = lin.solid_strains()
lin_bu = lin.bush_strains()
for name, d in [('bar_strains', lin_bs), ('solid_strains', lin_ss), ('bush_strains', lin_bu)]:
    for sc, df in d.items():
        print(f"{name} sc={sc}: {len(df)} rows, {df['EID'].nunique()} elements")


bar_strains sc=1: 832 rows, 416 elements
bar_strains sc=2: 832 rows, 416 elements
solid_strains sc=1: 19432 rows, 4858 elements
solid_strains sc=2: 19432 rows, 4858 elements
bush_strains sc=1: 4158 rows, 803 elements
bush_strains sc=2: 4158 rows, 803 elements


In [7]:
# §20g — Results helper: check all NL subcase 1 result types
r_nl = nl.results(1)
r_nl


Results(subcase=1, displacements(8102 rows), solid_stresses(19432 rows), bar_stresses(832 rows), element_forces(832 rows), spc_forces(8102 rows), bar_strains(832 rows), solid_strains(19432 rows), nl_bar_stresses(3328 rows), nl_bush_stresses(528 rows), nl_solid_stresses(24290 rows))

In [8]:
## §21 — OSTR1EL: Element-CS Strains & Bug Fix

# `OSTR1EL` is the element-coordinate-system strain output table, distinct from `OSTR1X` (basic/material CS).
# Previously `bar_strains()` inadvertently included both because `"OSTR1"` substring-matched `"OSTR1EL"`, doubling the row count.
# The fix claims `OSTR1EL` first so it is decoded separately.

In [9]:
# Bug-fix verification: bar_strains() now returns exactly 832 rows (no duplicates)
bs = lin.bar_strains()
for sc, df in bs.items():
    dups = df.duplicated(subset=["EID", "GRID", "SD"]).sum()
    print(f"bar_strains  sc={sc}: {len(df):>6} rows, {dups} duplicates")

bar_strains  sc=1:    832 rows, 0 duplicates
bar_strains  sc=2:    832 rows, 0 duplicates


In [10]:
# New element-CS strain methods on the linear model
bs_el = lin.bar_strains_el()
ss_el = lin.solid_strains_el()
bu_el = lin.bush_strains_el()
for sc in bs_el:
    print(f"bar_strains_el   sc={sc}: {len(bs_el[sc]):>6} rows")
for sc in ss_el:
    print(f"solid_strains_el sc={sc}: {len(ss_el[sc]):>6} rows")
for sc in bu_el:
    print(f"bush_strains_el  sc={sc}: {len(bu_el[sc]):>6} rows")

bar_strains_el   sc=1:    832 rows
bar_strains_el   sc=2:    832 rows
solid_strains_el sc=1:  19432 rows
solid_strains_el sc=2:  19432 rows
bush_strains_el  sc=1:   4158 rows
bush_strains_el  sc=2:   4158 rows


In [11]:
import pandas as pd

# Compare basic-CS vs element-CS bar strains for the same element
eid = bs[1]["EID"].iloc[0]
basic   = bs[1][bs[1]["EID"] == eid].assign(coord="basic")
element = bs_el[1][bs_el[1]["EID"] == eid].assign(coord="element")
pd.concat([basic, element]).set_index(["EID", "GRID", "SD", "coord"])

SXC           SXD           SXE           SXF  \
EID GRID SD  coord                                                             
1   7573 0.0 basic    1.277226e-07  1.554944e-07  1.277494e-07  9.997752e-08   
    225  1.0 basic    1.281082e-07  9.972634e-08  1.273638e-07  1.557456e-07   
    7573 0.0 element  1.277226e-07  1.554944e-07  1.277494e-07  9.997752e-08   
    225  1.0 element  1.281082e-07  9.972634e-08  1.273638e-07  1.557456e-07   

                              SMAX          SMIN          MS_T          MS_C  
EID GRID SD  coord                                                            
1   7573 0.0 basic    1.554944e-07  9.997752e-08  1.401298e-45  1.401298e-45  
    225  1.0 basic    1.557456e-07  9.972634e-08  1.401298e-45  1.401298e-45  
    7573 0.0 element  1.554944e-07  9.997752e-08  1.401298e-45  1.401298e-45  
    225  1.0 element  1.557456e-07  9.972634e-08  1.401298e-45  1.401298e-45

In [ ]:
## §22 — Normal Modes (`cylinder-0002.op2`)

# SOL 103 run with 10 extracted modes.  Available tables: `LAMA` (eigenvalues),
# `OUGV1` (eigenvectors), `OQG1` (SPC forces per mode), `OGPWG` (grid weight).

# **Bug fixed this session**: `decode_lama` was picking up the IDENT header record as
# the data payload, producing garbage (MODE=101, all zeros).  The fix adds content-
# based validation — a record only qualifies as eigenvalue data if its first word is a
# valid mode number (1–100000) and `sqrt(EIGENVALUE) ≈ RADIANS` within 5 %.

In [16]:
import importlib, op2_native, op2_native.decoders.lama as _lama_mod
importlib.reload(_lama_mod)
importlib.reload(op2_native.reader)
from op2_native.reader import OP2

NM_OP2 = 'run_files/model/cylinder-0002.op2'
nm = OP2(NM_OP2)

print("Tables in file:", nm.table_names(results_only=True))


Tables in file: ['OGPWG', 'OGPWG', 'OQG1', 'LAMA', 'LAMA', 'OUGV1']


In [17]:
# §22a — Eigenvalues (LAMA)
ev = nm.eigenvalues()
print(f"Eigenvalue subcases: {list(ev.keys())}  (one entry = all modes)")
ev[1]

Eigenvalue subcases: [1]  (one entry = all modes)


,MODE,ORDER,EIGENVALUE,RADIANS,CYCLES,GENM,GENSTIF
0,1,1,2.991190e+05,546.917725,87.044662,1.0,2.991190e+05
1,2,2,3.002813e+05,547.979248,87.213608,1.0,3.002813e+05
2,3,3,1.715525e+06,1309.780396,208.458023,1.0,1.715525e+06
3,4,4,2.779823e+06,1667.280029,265.355865,1.0,2.779823e+06
4,5,5,3.122773e+06,1767.136963,281.248566,1.0,3.122773e+06
5,6,6,3.123913e+06,1767.459473,281.299896,1.0,3.123913e+06
6,7,7,8.516311e+06,2918.271973,464.457397,1.0,8.516311e+06
7,8,8,9.876048e+06,3142.617920,500.163177,1.0,9.876048e+06
8,9,9,9.878351e+06,3142.984375,500.221527,1.0,9.878351e+06
9,10,10,1.178188e+07,3432.474121,546.295227,1.0,1.178188e+07


In [18]:
# §22b — Eigenvectors (OUGV1) — one subcase per mode
eig_vecs = nm.displacements()
print(f"Eigenvector modes: {list(eig_vecs.keys())}  ({len(eig_vecs[1])} DOFs each)")

# Spot-check mode 1, GRID 1 against F06
g1 = eig_vecs[1][eig_vecs[1]['GRID'] == 1].iloc[0]
print(f"\nMode 1  GRID 1  (F06 reference in brackets):")
print(f"  T1={g1.DX:+.6e}  [-1.822885E-01]")
print(f"  T2={g1.DY:+.6e}  [-6.204547E-02]")
print(f"  T3={g1.DZ:+.6e}  [-8.168588E-02]")
print(f"  R1={g1.RX:+.6e}  [-1.389719E-02]")
print(f"  R2={g1.RY:+.6e}  [+4.082382E-02]")
print(f"  R3={g1.RZ:+.6e}  [-2.415273E-06]")

Eigenvector modes: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]  (8102 DOFs each)

Mode 1  GRID 1  (F06 reference in brackets):
  T1=-1.822885e-01  [-1.822885E-01]
  T2=-6.204547e-02  [-6.204547E-02]
  T3=-8.168588e-02  [-8.168588E-02]
  R1=-1.389719e-02  [-1.389719E-02]
  R2=+4.082382e-02  [+4.082382E-02]
  R3=-2.415273e-06  [-2.415273E-06]


In [19]:
# §22c — Results helper summary
r_nm = nm.results(1)
r_nm

Results(subcase=1, displacements(8102 rows), spc_forces(8102 rows), eigenvalues(10 rows))

In [30]:

# §23a — Load with geometry, inspect grids + connectivity
import importlib, op2_native.decoders.geom_dat as _gm, op2_native.reader as _rd
importlib.reload(_gm); importlib.reload(_rd)
from op2_native.reader import OP2

LIN_OP2 = 'run_files/model/cylinder-0000.op2'
lin_g = OP2(LIN_OP2, geometry=True)   # auto-finds cylinder-0000.dat

grids = lin_g.grid_coordinates()
conn  = lin_g.element_connectivity()

print(f"Grids: {len(grids):,}  (first 3)")
print(grids.head(3).to_string(index=False))
print()
for et, df in conn.items():
    print(f"  {et}: {len(df):,} elements  cols={list(df.columns)}")


Grids: 8,102  (first 3)
 GID        X        Y   Z  CP  CD
   1 2.000000 0.000000 0.0   0   0
   2 1.961571 0.390181 0.0   0   0
   3 1.847759 0.765367 0.0   0   0

  CBEAM: 416 elements  cols=['EID', 'PID', 'GA', 'GB']
  CBUSH: 528 elements  cols=['EID', 'PID', 'GA', 'GB']
  CTETRA: 4,858 elements  cols=['EID', 'PID', 'G1', 'G2', 'G3', 'G4']


In [31]:

# §23b — Element centroids (average of corner-node coordinates)
centroids = lin_g.element_centroids('CTETRA')
print(f"CTETRA centroids: {len(centroids):,}")
print(centroids.head(5).to_string(index=False))


CTETRA centroids: 4,858
 EID         X         Y        Z
1057 -0.115220 -1.186168 3.529812
1058 -0.029538 -1.251565 3.755318
1059  0.001556 -0.957993 3.498782
1060 -0.006497 -1.093900 3.446186
1061 -0.974635 -1.322192 3.152793


In [33]:

# §23c — Join stresses to centroids: top-10 worst VM elements + their location
import pandas as pd

stresses = lin_g.solid_stresses()[1]          # subcase 1
cent_df  = lin_g.element_centroids('CTETRA')  # EID, X, Y, Z

# solid_stresses has corner output; take centroid row per element (max abs VM)
vm_col = [c for c in stresses.columns if 'VM' in c][0]
cen_vm = stresses.groupby('EID')[vm_col].max().reset_index()

top10 = (
    cen_vm.merge(cent_df, on='EID')
    .nlargest(10, vm_col)
    [['EID', vm_col, 'X', 'Y', 'Z']]
    .reset_index(drop=True)
)
print(f"Top-10 {vm_col} stress elements with centroid coordinates:")
print(top10.to_string(index=False))


Top-10 VM stress elements with centroid coordinates:
 EID        VM         X         Y        Z
4350 35.574055  0.262125 -0.623054 3.887830
2152 35.021530 -0.002864 -0.668883 3.887983
1153 33.291714 -0.264558 -0.629489 3.926229
3065 32.942745  0.476966 -0.471967 3.958686
5059 32.461941 -0.105499 -0.688485 3.814213
3450 32.253296  0.620503 -0.253532 3.930360
2040 31.353456  0.084269 -0.611258 3.887983
4726 30.976254  0.322133 -0.538185 3.887830
3064 30.754147  0.176154 -0.670904 3.775813
1854 30.667685  0.420175 -0.553286 3.846515
